HAND WRITTEN TEXT GENERATION

In [5]:
!ls

'archive (2).zip'   sample_data   words.txt


In [6]:
with open("words.txt", "r", encoding="utf-8") as f:
    data = f.readlines()

clean_text = ""

for line in data:
    if not line.startswith("#"):
        parts = line.strip().split()
        if len(parts) >= 9:   # important condition
            word = parts[-1]
            clean_text += word + " "

print(clean_text[:500])

A MOVE to stop Mr. Gaitskell from nominating any more Labour life Peers is to be made at a meeting of Labour Ps tomorrow . Mr. Michael Foot has put down a resolution on the subject and he is to be backed by Mr. Will Griffiths , P for Manchester Exchange . A MOVE to stop Mr. Gaitskell from nominating any more Labour life Peers is to be made at a meeting of Labour Ps tomorrow . Mr. Michael Foot has put down a resolution on the subject and he is to be backed by Mr. Will Griffiths , P for Manchester


In [7]:
text = clean_text.lower()
text = text[:50000]   # limit for faster training

In [8]:
import numpy as np
import tensorflow as tf

chars = sorted(list(set(text)))
char_to_idx = {c:i for i,c in enumerate(chars)}
idx_to_char = {i:c for i,c in enumerate(chars)}

vocab_size = len(chars)
print("Vocabulary Size:", vocab_size)

Vocabulary Size: 49


In [9]:
sequence_length = 40
X = []
y = []

for i in range(len(text) - sequence_length):
    seq = text[i:i+sequence_length]
    target = text[i+sequence_length]
    X.append([char_to_idx[c] for c in seq])
    y.append(char_to_idx[target])

X = np.array(X)
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

X = X.reshape((X.shape[0], X.shape[1], 1))
X = X / float(vocab_size)

print("Shape of X:", X.shape)

Shape of X: (49960, 40, 1)


In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential([
    LSTM(256, return_sequences=True, input_shape=(sequence_length, 1)),
    Dropout(0.2),
    LSTM(256),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 40, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 49)             │        12,593 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 802,097 (3.06 MB)

 Trainable params: 802,097 (3.06 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(X, y, epochs=15, batch_size=64)

Epoch 1/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 323s 409ms/step - loss: 2.9914
Epoch 2/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 317s 403ms/step - loss: 2.7272
Epoch 3/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 318s 407ms/step - loss: 2.5872
Epoch 4/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 330s 422ms/step - loss: 2.5167
Epoch 5/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 329s 421ms/step - loss: 2.4318
Epoch 6/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 374s 411ms/step - loss: 2.3291
Epoch 7/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 318s 406ms/step - loss: 2.2221
Epoch 8/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 322s 406ms/step - loss: 2.1030
Epoch 9/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 319s 409ms/step - loss: 1.9804
Epoch 10/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 318s 407ms/step - loss: 1.8894
Epoch 11/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 319s 408ms/step - loss: 1.7815
Epoch 12/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 321s 410ms/step - loss: 1.6767
Epoch 13/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 316s 405ms/step - loss: 1.5831
Epoch 14/15
781/781 ━━━━━━━━━━━━━━━━━━━━ 320s 402ms/step - loss: 1.5011
E

In [13]:
import random

start_index = random.randint(0, len(X)-1)
pattern = X[start_index]

generated = ""

for i in range(300):
    prediction = model.predict(pattern.reshape(1, sequence_length, 1), verbose=0)
    index = np.argmax(prediction)
    result = idx_to_char[index]

    generated += result

    pattern = np.vstack((pattern[1:], [[index / vocab_size]]))

print("Generated Text:\n")
print(generated)

Generated Text:

re to amr wiet the common corcses of she common market s collont . the sistice and is his to america . and the past week . mr. brown sennnted that the prised of angrican salks with the prlicc of acricans thet there is not of the pro-communist . and mr. kaunda caahar pe the proiecr tarkinga wiat the 
